# Preparação de Dados para Machine Learning — YouTube com PySpark

Neste notebook enriquecemos o dataset de vídeos do YouTube com novos campos derivados, aplicamos transformações necessárias para modelos de ML (indexação, vetorização, normalização e PCA) e treinamos um modelo de Regressão Linear para estimar a quantidade de comentários.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType
from pyspark.ml.feature import StringIndexer, VectorAssembler, Normalizer, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder \
    .appName('YouTube - Preparacao de Dados') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print(f'SparkSession iniciada | Spark {spark.version}')

## Passo 1 — Leitura do arquivo `videos-tratados.snappy.parquet`

In [ ]:
# Passo 1: Leitura do arquivo parquet no dataframe df_video
df_video = spark.read.parquet('videos-tratados.snappy.parquet')

print('Esquema do df_video:')
df_video.printSchema()
print(f'Total de registros: {df_video.count()}')
df_video.show(5, truncate=True)

## Passo 2 — Adição da coluna `Month` extraída de `Published At`

In [ ]:
# Passo 2: Adicionar coluna 'Month' com o mês da coluna 'Published At'
# Garantimos que 'Published At' seja do tipo date antes de extrair o mês
df_video = df_video.withColumn(
    'Published At',
    F.to_date(F.col('Published At'))
)

df_video = df_video.withColumn(
    'Month',
    F.month(F.col('Published At'))
)

print('Coluna Month adicionada:')
df_video.select('Published At', 'Month').show(8)

## Passo 3 — Adição da coluna `Keyword Index` (StringIndexer)

In [ ]:
# Passo 3: Transformar a coluna 'Keyword' em valores numéricos com StringIndexer
indexer = StringIndexer(
    inputCol='Keyword',
    outputCol='Keyword Index',
    handleInvalid='keep'
)

df_video = indexer.fit(df_video).transform(df_video)

print('Coluna Keyword Index adicionada:')
df_video.select('Keyword', 'Keyword Index').distinct().orderBy('Keyword Index').show(15)

## Passo 4 — Criação do vetor `Features` com VectorAssembler

In [ ]:
# Passo 4: Criar vetor 'Features' com os campos numéricos selecionados
# O campo 'Year' foi lido como string — convertemos para inteiro antes do assembler
df_video = df_video.withColumn('Year', F.col('Year').cast(IntegerType()))

# VectorAssembler aceita apenas campos numéricos
assembler = VectorAssembler(
    inputCols=['Likes', 'Views', 'Year', 'Month', 'Keyword Index'],
    outputCol='Features',
    handleInvalid='keep'
)

df_video = assembler.transform(df_video)

print('Vetor Features criado:')
df_video.select('Likes', 'Views', 'Year', 'Month', 'Keyword Index', 'Features').show(5, truncate=True)

## Passo 5 — Adição da coluna `Features Normal` (Normalização)

In [ ]:
# Passo 5: Normalizar o vetor 'Features' — remover nulos antes de normalizar
# Normalizer usa norma L2 por padrão
df_video = df_video.dropna(subset=['Features'])

normalizer = Normalizer(
    inputCol='Features',
    outputCol='Features Normal',
    p=2.0  # norma L2
)

df_video = normalizer.transform(df_video)

print('Coluna Features Normal adicionada:')
df_video.select('Features', 'Features Normal').show(5, truncate=True)

## Passo 6 — Adição da coluna `Features PCA` (redução de 5 → 1 componente)

In [ ]:
# Passo 6: Reduzir o vetor de 5 características para 1 componente principal com PCA
pca = PCA(
    k=1,
    inputCol='Features Normal',
    outputCol='Features PCA'
)

pca_model = pca.fit(df_video)
df_video = pca_model.transform(df_video)

print('Variância explicada pelo componente principal:', pca_model.explainedVariance)
print('\nColuna Features PCA adicionada:')
df_video.select('Features Normal', 'Features PCA').show(5, truncate=True)

## Passo 7 — Separação em conjuntos de Treino (80%) e Teste (20%)

In [ ]:
# Passo 7: Separar df_video em 80% treino e 20% teste
df_treino, df_teste = df_video.randomSplit([0.8, 0.2], seed=42)

print(f'Registros para treino: {df_treino.count()}')
print(f'Registros para teste:  {df_teste.count()}')
print(f'Total:                 {df_video.count()}')

## Passo 8 — Modelo de Regressão Linear para estimar `Comments`

In [ ]:
# Passo 8: Criar e avaliar um modelo de Regressão Linear
# Label: campo 'Comments' | Features: 'Features Normal'

# O campo Comments precisa ser numérico (double) para o modelo
df_treino = df_treino.withColumn('Comments', F.col('Comments').cast('double'))
df_teste  = df_teste.withColumn('Comments',  F.col('Comments').cast('double'))

# Remover nulos na coluna label
df_treino = df_treino.dropna(subset=['Comments'])
df_teste  = df_teste.dropna(subset=['Comments'])

# Treinamento
lr = LinearRegression(
    featuresCol='Features Normal',
    labelCol='Comments',
    maxIter=50,
    regParam=0.1,
    elasticNetParam=0.0
)

lr_model = lr.fit(df_treino)

# Predição no conjunto de teste
predicoes = lr_model.transform(df_teste)

# Avaliação
evaluator_rmse = RegressionEvaluator(
    labelCol='Comments',
    predictionCol='prediction',
    metricName='rmse'
)
evaluator_r2 = RegressionEvaluator(
    labelCol='Comments',
    predictionCol='prediction',
    metricName='r2'
)
evaluator_mae = RegressionEvaluator(
    labelCol='Comments',
    predictionCol='prediction',
    metricName='mae'
)

rmse = evaluator_rmse.evaluate(predicoes)
r2   = evaluator_r2.evaluate(predicoes)
mae  = evaluator_mae.evaluate(predicoes)

print('=== Avaliação do Modelo de Regressão Linear ===')
print(f'  RMSE (Erro Quadrático Médio Raiz): {rmse:.2f}')
print(f'  MAE  (Erro Absoluto Médio):        {mae:.2f}')
print(f'  R²   (Coeficiente de Determinação):{r2:.4f}')
print()
print(f'  Intercepto:   {lr_model.intercept:.4f}')
print(f'  Coeficientes: {lr_model.coefficients}')
print()

# Visualizar predições vs valores reais
print('Amostra de predições vs valores reais:')
predicoes.select('Comments', 'prediction').show(10)

## Passo 9 — Salvar `df_video` como `videos-preparados-parquet`

In [ ]:
# Passo 9: Salvar o dataframe df_video no formato parquet com cabeçalho
df_video.write \
    .option('header', 'true') \
    .mode('overwrite') \
    .parquet('videos-preparados-parquet')

print('df_video salvo como videos-preparados-parquet com sucesso!')
print(f'Colunas finais ({len(df_video.columns)}):')
for col in df_video.columns:
    print(f'  - {col}')

In [ ]:
# Encerramento da SparkSession
spark.stop()
print('SparkSession encerrada com sucesso!')